In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# Константы
MODEL_NAME = 'ai-forever/ruRoberta-large'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 16

### Читаем данные

In [ ]:
# Читаем доступные категории
with open('data/categories.txt', 'r') as f:
    categories = f.readlines()
for i in range(len(categories)):
    categories[i] = categories[i].replace('\n', '')
map_categories = {}
inverse_map = {}
for i in range(len(categories)):
    map_categories[categories[i]] = i
    inverse_map[i] = categories[i]
print(map_categories)
print(inverse_map)

In [ ]:
# Читаем данные которые надо разметить
test_data = pd.read_csv('data/test.csv')
test_data.head(5)

### Подготавливаем данные

In [ ]:
# Загружаем токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(categories), dtype='auto', device_map='auto')
state_dict = torch.load('output_files/best_model.pt', map_location=DEVICE)
model.load_state_dict(state_dict)

In [ ]:
# Датасет отзывов
class ReviewDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

# Подготовка батча перед подачей в модель
def collate_fn(batch, tokenizer, device):
    reviews = [review for review in batch]
    input_ids = [tokenizer(review, add_special_tokens=True, return_tensors='pt')['input_ids'].reshape(-1) for review in reviews]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id).to(device)
    attention_mask = (input_ids != tokenizer.pad_token_id).long().to(device)
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask
    }

In [ ]:
# Создаем датасет и даталоадер
test_dataset = ReviewDataset(test_data['text'].to_list())
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda x: collate_fn(x, tokenizer, DEVICE))

### Размечаем данные

In [ ]:
model.eval()
all_preds = []
for batch in tqdm(test_dataloader, total=len(test_dataloader), desc='Processing dataset'):
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    preds = torch.argmax(logits, dim=-1)
    preds_cat = [inverse_map[i] for i in preds.tolist()]
    all_preds.extend(preds_cat)

### Сохраняем результат

In [ ]:
result = pd.DataFrame({'text': test_data['text'].to_list(), 'category': all_preds})
result.to_csv('output_files/submission.csv', index=False)